In [ ]:
#install rdflib to access RDF data
pip install rdflib

In [1]:
#upload RDF ontology
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, RDFS, OWL
# assign namespaces 
atticlaw = Namespace('http://www.ontologia.fr/OTB/atticlaw/')
orators = Namespace('http://www.ontologia.fr/OTB/classical_orators#')
processes = Namespace('http://ontologia.fr/OTB/legal_processes#')
otv = Namespace('http://www.ontologia.fr/OTB/otv#')
# create graph and parse ontology file
g = Graph()
g.parse("attic_law2.1.ttl", format="turtle")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\attic_law2.1.ttl'

In [ ]:
# query for basic person information
# this query finds all the people in the ontology, as well as attributes such as gender, labels (in English and Greek), and external identifiers
person_query = """
SELECT ?person ?gender ?label_English ?label_Greek ?PAA ?wikidata ?type
WHERE {
    {?person a atticlaw:Man} UNION {?person a atticlaw:Woman}
    ?person a ?gender ;
            rdfs:label ?label_English .
    FILTER(?gender != owl:NamedIndividual)
    FILTER(lang(?label_English)="en")
    OPTIONAL { ?person rdfs:label ?label_Greek . FILTER(lang(?label_Greek)!="en") }
    OPTIONAL { ?person atticlaw:PAA ?PAA }
    OPTIONAL { ?person atticlaw:wikidata ?wikidata }
    OPTIONAL { ?person a atticlaw:Mythological_Person BIND(atticlaw:Mythological_Person as ?type) }
}
"""
# put query results into an index stored as a dictionary
person_index = {}
for row in g.query(person_query):
    person_uri = row.person
    name_en = row.label_English
    name_grc = row.label_Greek
    PAA = row.PAA
    gender = row.gender
    person_type = row.type
    wikidata = row.wikidata
    
    if person_uri not in person_index:
        person_index[person_uri]={
            "name_en": name_en,
            "name_grc" : name_grc,
            "PAA" : PAA,
            "gender" : gender,
            "wikidata": wikidata,
            "type": person_type,
            "event_roles" : set(),
            "speeches" : set(),
            "authored_speeches" : set(),
            "relations" : set()
        }


In [ ]:
# query for legal event information
# this query returns the legal roles for each person
for person_uri in person_index.keys():
    legal_event_query = f"""
    SELECT ?person ?legalrole ?event ?eventLabel
    WHERE
    {{
        VALUES ?person {{ <{person_uri}> }}
        ?suit rdfs:label "lawsuit"@en .
        ?eventType rdfs:subClassOf+ ?suit .
        ?event a ?eventType ;
               ?legalrole ?person .
        ?legalrole rdfs:subPropertyOf* atticlaw:hasParticipant .
        ?event rdfs:label ?eventLabel
    }}
    """
    # add query results to the person index
    for row in g.query(legal_event_query):
        person_uri = row.person
        legal_role = row.legalrole
        event = row.event
        event_label = row.eventLabel
        event_tuple = (event, event_label, legal_role)
        person_index[person_uri]["event_roles"].add(event_tuple)



In [ ]:
# query for speech information
# this query returns the speaking roles for each person
for person_uri in person_index.keys():
    speech_query = f"""
    SELECT ?person ?speech ?speechLabel
    WHERE
    {{
        VALUES ?person {{ <{person_uri}> }}
        ?speech atticlaw:hasSpeaker ?person ;
            rdfs:label ?speechLabel
    }}
    """
    # add query results to the person index
    for row in g.query(speech_query):
        person_uri = row.person
        speech = row.speech
        speech_label = row.speechLabel
        speech_tuple = (speech, speech_label)
        person_index[person_uri]["speeches"].add(speech_tuple)


In [ ]:
# query for authorship information
# this query returns the authorship roles for each person, with varying levels of certainty
for person_uri in person_index.keys():
    author_query = f"""
    SELECT ?person ?authoredSpeech ?authoredSpeechLabel ?authorship
    WHERE
    {{
        VALUES ?person {{ <{person_uri}> }}
        VALUES ?authorship {{ orators:hasAuthor orators:hasProbableAuthor orators:hasImprobableAuthor }}
        ?authoredSpeech ?authorship ?person ;
                    rdfs:label ?authoredSpeechLabel.
        FILTER(STRSTARTS(STR(?authoredSpeech), "http://www.ontologia.fr/OTB/classical_orators#"))
    }}
    """
    # add query results to the person index
    for row in g.query(author_query):
        person_uri = row.person
        authored_speech = row.authoredSpeech
        authored_speech_label = row.authoredSpeechLabel
        authorship = row.authorship
        author_tuple = (authored_speech, authored_speech_label, authorship)
        person_index[person_uri]["authored_speeches"].add(author_tuple)


In [ ]:
# query for relations
# this query finds the familial relations for each person
for person_uri in person_index.keys():
    relations_query = f"""
    SELECT ?person ?relation ?person2
    WHERE
    {{
        VALUES ?relation {{atticlaw:parentOf atticlaw:childOf atticlaw:siblingOf atticlaw:wifeOf atticlaw:husbandOf}}
        VALUES ?person {{<{person_uri}>}}
        ?person ?relation ?person2 .      
    }}
    """
    # add query results to the person index
    for row in g.query(relations_query):
        person_uri = row.person
        relation = row.relation
        relative = row.person2
        relation_tuple = (relation, relative)
        person_index[person_uri]["relations"].add(relation_tuple)

In [ ]:
# Because these queries can take a long time to run, here you can save the query result index locally
from types import resolve_bases
import pickle

with open('person_index.pkl', 'wb') as file:
    pickle.dump(person_index, file)

In [ ]:
# load the saved query result
from types import resolve_bases
import pickle

with open('person_index.pkl', 'rb') as file:
    person_index = pickle.load(file)

print(person_index)

In [ ]:
# sort the query result index in order of URI
sorted_people = dict(sorted(person_index.items()))
print(sorted_people)

In [ ]:
# create html person index page
# the index includes a JavaScript search/filter function, allowing people to be searched for by name and filtered by gender and legal roles
# each person in the index is displayed on a card, presented in a grid layout
person_html_template = """---
title: Person Index
layout: page
permalink: /person-index
---
<!DOCTYPE html>
<head>
    <meta charset="UTF-8">
    <title>Person Index</title>
    <style>
        body {{ font-family: "New Athena Unicode", "Cardo", "Palatino Linotype", serif; }}
            .person-index {{ max-width: 800px; margin: 0 auto; padding: 1rem; }}
            .person-card {{ 
                border: 1px solid #e0e0e0; 
                margin: 1.5rem 0; 
                padding: 1.5rem;
                border-radius: 8px;
                background: #fafafa;
            }}
            .name-en {{
                font-size: 1.8em; 
                font-weight: bold; 
                margin: 0 0 0.5rem 0;
                color: #2c3e50;
            }}
            .name-grc {{ 
                font-style: italic; 
                color: #7f8c8d;
                font-size: 1.2em;
                margin: 0 0 .3rem 0;
            }}
            .index-header {{ 
                text-align: center; 
                margin-bottom: 1rem;
                font-size: 1.2em;
                border-bottom: 2px solid #34495e;
                padding-bottom: 1rem;
            }}
            .person-grid {{
                display: grid;
                grid-template-columns: repeat(auto-fill, minmax(250px, 1fr));
                gap: 20px;
                margin-top: 2rem;
            }}
            .container {{
                display: grid;
                grid-template-columns: 280px 1fr;  /* 300px for facet panel, rest for content */
                gap: 20px;
                max-width: 1400px;
                margin: 0 auto;
                padding: 20px;
            }}
            .main-content {{
                background: white;
                border-radius: 8px;
                padding: 20px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }}
            /* Facets Panel */
            .facets-panel {{
                background: white;
                border-radius: 8px;
                padding: 20px;
                box-shadow: 0 2px 4px rgba(0,0,0,0.1);
                height: fit-content;
                position: sticky;
                top: 20px;
            }}
            .facets-group {{
                margin-bottom: 25px;
            }}
            .facet-option {{
                display: flex;
                align-items: center;
                margin-bottom: 8px;
                cursor: pointer;
                padding: 6px 8px;
                border-radius: 4px;
                transition: background 0.2s;
            }}
            .facet-option:hover {{
                background: #f0f0f0;
            }}
            .facet-option input[type="checkbox"] {{
                margin-right: 10px;
                cursor: pointer;
                width: 16px;
                height: 16px;
            }}
        </style>
</head>
<body>
    <div class="container">
        <div class="facets-panel">
            <h2>Filter Persons</h2>
            <div class="facet-group">
                <h3>Name Search</h3>
                <input type="text" id="searchBox" placeholder="Search English or Greek...">
            </div>
            <div class="facet-group">
                <h3>Gender</h3>
                <div id="facet-gender">
                    <label class="facet-option">
                        <input type="checkbox" data-facet="gender" data-value="Male">
                        <span>Male</span>
                    </label>
                    <label class="facet-option">
                        <input type="checkbox" data-facet="gender" data-value="Female">
                        <span>Female</span>
                    </label>
                </div>
            </div>
            <div class="facet-group">
                <h3>Legal Roles</h3>
                <div id="facet-roles">
                    <label class="facet-option">
                        <input type="checkbox" data-facet="role" data-value="plaintiff">
                        <span>Plaintiff</span>
                    </label>
                    <label class="facet-option">
                        <input type="checkbox" data-facet="role" data-value="defendant">
                        <span>Defendant</span>
                    </label>
                    <label class="facet-option">
                        <input type="checkbox" data-facet="role" data-value="claimant">
                        <span>Claimant</span>
                    </label>
                    <label class="facet-option">
                        <input type="checkbox" data-facet="role" data-value="witness">
                        <span>Witness</span>
                    </label>
                    <label class="facet-option">
                        <input type="checkbox" data-facet="role" data-value="speaker">
                        <span>Speaker</span>
                    </label>
                </div> 
            </div>
        </div>
    <div class="main-content">
        <div class='person-index'>
            <div class='index-header'>
                <p>Total person entities: <span id="total-count">0</span></p>
                <p>Displayed person entities: <span id="displayed-count">0</span></p>
            </div>
        <div class='person-grid'>
            {person_cards}
        </div>
        </div>
        </div>
        </div>
<script>
// search function
// load page
document.addEventListener('DOMContentLoaded', function() {{

    // get all cards
    const personCards = document.querySelectorAll('.person-card');
    const totalCards = document.querySelectorAll('.person-card').length;
    document.getElementById('total-count').textContent = totalCards;
    document.getElementById('displayed-count').textContent = totalCards;

    //get gender checkboxes
    const maleCheckbox = document.querySelector('input[data-facet="gender"][data-value="Male"]');
    const femaleCheckbox = document.querySelector('input[data-facet="gender"][data-value="Female"]');
    
    //get search box
    const searchBox = document.getElementById('searchBox');

    //get legal role boxes
    const plaintiffCheckbox = document.querySelector('input[data-facet="role"][data-value="plaintiff"]');
    const defendantCheckbox = document.querySelector('input[data-facet="role"][data-value="defendant"]');
    const claimantCheckbox = document.querySelector('input[data-facet="role"][data-value="claimant"]');
    const witnessCheckbox = document.querySelector('input[data-facet="role"][data-value="witness"]');
    const speakerCheckbox = document.querySelector('input[data-facet="role"][data-value="speaker"]');


    // apply all filters
    function applyFilters() {{

        // current filter states
        const showMale = maleCheckbox.checked;
        const showFemale = femaleCheckbox.checked;
        const searchTerm = searchBox.value.toLowerCase();
        const showPlaintiff = plaintiffCheckbox.checked;
        const showDefendant = defendantCheckbox.checked;
        const showClaimant = claimantCheckbox.checked;
        const showWitness = witnessCheckbox.checked;
        const showSpeaker = speakerCheckbox.checked;

        let visibleCount = 0;

        // loop through each card
        personCards.forEach(card => {{
            // Check gender filter
            let genderMatch = true;
            if (showMale || showFemale) {{ // Only apply gender filter if at least one is checked, else show all
                const cardGender = card.getAttribute('gender');
                genderMatch = (showMale && cardGender === 'Male') || (showFemale && cardGender === 'Female');
            }}

            // Check search filter
            const cardText = card.textContent.toLowerCase();
            const searchMatch = cardText.includes(searchTerm);

            // check role filter
            let roleMatch = true;
            if (showPlaintiff || showDefendant || showClaimant || showWitness || showSpeaker) {{
                const cardRolePlaintiff = card.getAttribute('plaintiff');
                const cardRoleDefendant = card.getAttribute('defendant');
                const cardRoleClaimant = card.getAttribute('claimant');
                const cardRoleWitness = card.getAttribute('witness');
                const cardRoleSpeaker = card.getAttribute('speaker');
                
                if (showPlaintiff && cardRolePlaintiff !== "True") roleMatch = false;
                if (showDefendant && cardRoleDefendant !== "True") roleMatch = false;
                if (showClaimant && cardRoleClaimant !== "True") roleMatch = false;
                if (showWitness && cardRoleWitness !== "True") roleMatch = false;
                if (showSpeaker && cardRoleSpeaker !== "True") roleMatch = false;

            }}

            // Show card only if BOTH filters pass
            if (genderMatch && searchMatch && roleMatch) {{
                card.style.display = 'block';
                visibleCount++;
            }} else {{
                card.style.display = 'none';
            }}
        }});
        document.getElementById('displayed-count').textContent = visibleCount;
    }}
    
    // Add event listeners
    maleCheckbox.addEventListener('change', applyFilters);
    femaleCheckbox.addEventListener('change', applyFilters);
    searchBox.addEventListener('input', applyFilters);
    plaintiffCheckbox.addEventListener('change', applyFilters);
    defendantCheckbox.addEventListener('change', applyFilters);
    claimantCheckbox.addEventListener('change', applyFilters);
    witnessCheckbox.addEventListener('change', applyFilters);
    speakerCheckbox.addEventListener('change', applyFilters);
}});
</script>
</body>
</html>
"""
# parse the person index to extract string values for the HTML index page
# store cards in a list
person_cards = []

# iterate through all people
for person in sorted_people.keys():
    # create unique links in the webpage using person URIs
    person_link = "/person-index/" + person.split('/')[-1].split('#')[-1]
    # extract person names in ancient Greek and English
    person_name_grc = sorted_people[person]["name_grc"]
    if person_name_grc == None:
        person_name_grc = ""
    person_name_en = sorted_people[person]["name_en"].title()
    # convert gender type into a string ("Male" or "Female")
    if sorted_people[person]["gender"].split('/')[-1] == 'Man':
        gender = "Male"
    else:
        gender = "Female"

    # set event roles as False
    plaintiff = False
    defendant = False
    claimant = False
    witness = False 
    speaker = False

    # if person has held any legal roles, change the attribute to True
    for event in sorted_people[person]["event_roles"]:
        event_role = event[2].split('/')[-1]
        if event_role == 'hasDefendant':
            defendant = True
        if event_role == 'hasPlaintiff':
            plaintiff = True
        if event_role == 'hasClaimant':
            claimant = True
        if event_role == 'hasWitness':
            witness = True
    if sorted_people[person]["speeches"] != set():
        speaker = True

    # create person card using values extracted previously
    person_card = f"""
    <div class="person-card" id="{person.split('/')[-1]}" gender="{gender}" plaintiff="{plaintiff}" defendant="{defendant}" claimant="{claimant}" witness="{witness}" speaker="{speaker}">
        <a href="{person_link}">
        <div class="name-grc">{person_name_grc}</div>
        <div class="name-en">{person_name_en}</div>
        </a>
    </div>
    """
    # add to the person card list
    person_cards.append(person_card)

# add person card list to the HTML template
person_html = person_html_template.format(
    count=len(person_index.keys()),
    person_cards="\n".join(person_cards))

with open("person_index.html", 'w', encoding='utf-8') as f:
    f.write(person_html)

In [ ]:
# make a unique page for each person in the dataset
import os
os.makedirs("persons", exist_ok=True)
person_page_template = """---
layout: page
title: {name_en}
permalink: {person_link}
exclude: true
---
<!DOCTYPE html>
<head>
    <meta charset="UTF-8">
    <style>
        body {{ font-family: "New Athena Unicode", "Cardo", "Palatino Linotype", serif; }}
        .name-grc {{ 
                font-style: italic; 
                color: #34495e;
                font-size: 1.7em;
                margin: 0 0 .1rem 0;
                text-align: center;
                border-bottom: 2px solid #34495e;
                padding-bottom: 1rem;
            }}
        .person-meta {{ 
                color: #131313; 
                margin: 0.5rem 0;
                font-size: 1.3em;
            }}
        .snippet {{
            font-size: 1.4em;
            color: rgb(44, 44, 44);
        }}
        .reference-attribution{{
            text-align: right;
        }}
        .reference-item {{
            border: solid 1px #899db2;
            border-radius: 5px;
            background-color: #edeff1;
            margin: 0em 0em;
            padding: .5em 1em;
        }}
        .collapsible {{
            border: 1px solid #e0e0e0; 
            margin: .5rem 0 0 0; 
            padding: 1rem;
            border-radius: 8px;
            background: #fafafa;
            cursor: pointer;
            width: 100%;
            text-align: left;
            outline: none;
            font-size: 1.3em;
            font-family: "New Athena Unicode", "Cardo", "Palatino Linotype", serif;
        }}
        .collapsible:hover {{
            background-color: #6b8195;
        }}
        .content {{
            display: none;
            overflow: hidden;
        }}
        .relation-list {{
            padding-left: 10px;
        }}
    </style>
</head>
<body>
    <div class="name-grc">{name_grc}</div>
    <div class="person-meta">Type: {real_or_fictional}</div>
    <div class="person-meta">Gender: {gender}</div>
    {external_id}
    {relations_section}
    {roles_section}
    {speech_section}
    {author_section}
    <script>
        var coll = document.getElementsByClassName("collapsible");
        var i;

        for (i = 0; i < coll.length; i++) {{
          coll[i].addEventListener("click", function() {{
            this.classList.toggle("active");
            var content = this.nextElementSibling;
            if (content.style.display === "block") {{
              content.style.display = "none";
            }} else {{
              content.style.display = "block";
            }}
          }});
        }}
    </script>
</body>
"""
for person in sorted_people.keys():
    person_uri = person
    person_data = sorted_people[person_uri]
    
    #gender attribute
    if person_data["gender"].split('/')[-1] == 'Man':
        gender = "Male"
    else:
        gender = "Female"
        
    #real or mythological attribute
    if person_data["type"] != None:
        real_or_fictional = "Mythological"
    else: 
        real_or_fictional = "Real"
        
    #identifier attribute
    external_id = ""
    if person_data["PAA"] != None:
        external_id += f"<div class='person-meta'>PAA: {person_data["PAA"]}</div>"
    if person_data["wikidata"] != None:
        external_id += f"<div class='person-meta'>Wikidata: <a href='https://www.wikidata.org/wiki/{person_data["wikidata"]}'>{person_data["wikidata"]}</a></div>"
    
    #event role attributes
    if person_data["event_roles"] != set():
        role_list = ""
        for event in person_data["event_roles"]:
            event_id = event[0]
            event_label = event[1]
            event_role = event[2]
            event_link = event_id.split('/')[-1]
            if event_role.split('/')[-1] == 'hasDefendant':
                legal_role = "Defendant"
            if event_role.split('/')[-1] == 'hasPlaintiff':
                legal_role = "Plaintiff"
            if event_role.split('/')[-1] == 'hasClaimant':
                legal_role = "Claimant"
            if event_role.split('/')[-1] == 'hasWitness':
                legal_role = "Witness"
            role_statement = f"<li id='{event_id}'>{legal_role} in: <a href='/event-index/{event_link}'>{event_label}</a></li>"
            role_list += role_statement
        roles_section = f"""
        <div class="person-meta">Legal Roles:</div>
        <ul>
        {role_list}
        </ul>
        """
    else:
        roles_section = ""

    #speaker attributes
    if person_data["speeches"] != set():
        speech_list = ""
        for speech in person_data["speeches"]:
            speech_id = speech[0]
            speech_label = speech[1]
            speech_statement = f"<li id='{speech_id}'>Speaker of <a href='/corpus-index/{speech_id.split('#')[-1].replace('.', "")}'>{speech_label}</a></li>"
            speech_list += speech_statement
        speech_section = f"""
        <div class="person-meta">Speaking Roles:</div>
        <ul>
        {speech_list}
        </ul>
        """
    else:
        speech_section = ""

    #authorship attributes
    if person_data["authored_speeches"] != set():
        authored_list = ""
        for authored in person_data["authored_speeches"]:
            authored_speech = authored[0] 
            authored_speech_label = authored[1] 
            authorship = authored[2]
            if authorship.split('#')[-1] == "hasAuthor":
                authorship_certainty = "Author of"
            if authorship.split('#')[-1] == "hasProbableAuthor":
                authorship_certainty = "Probable author of"
            if authorship.split('#')[-1] == "hasImprobableAuthor":
                authorship_certainty = "Improbable author of"
            if authorship.split('#')[-1] == "canonAuthor":
                authorship_certainty = "Canon attributed author of"
            author_statement = f"<li id='{authored_speech}'>{authorship_certainty} <a href='/corpus-index/{authored_speech.split('#')[-1].replace('.', '')}'>{authored_speech_label}</a></li>"
            authored_list += author_statement
        author_section = f"""
        <div class="person-meta">Authorship Roles:</div>
        <ul>
        {authored_list}
        </ul>
        """
    else:
        author_section = ""
        
    #relations section
    if person_data["relations"] != set():
        relations_section = "<div class='person-meta'>Relations:</div><ul class='relation-list'>"
        for relations in person_data["relations"]:
            relation = relations[0].split('/')[-1]
            relative = relations[1]
            relative_id = relative.split('/')[-1]
            relative_id = relative_id.split('#')[-1]
            relative_label = person_index[relative]["name_en"]
            if relation == "childOf":
                if gender == "Male":
                    relation_type = "Son of:"
                if gender == "Female":
                    relation_type = "Daughter of:"
            if relation == "parentOf":
                if gender == "Male":
                    relation_type = "Father of:"
                if gender == "Female":
                    relation_type = "Mother of:"
            if relation == "husbandOf":
                relation_type = "Husband of:"
            if relation == "wifeOf":
                relation_type = "Wife of:"
            if relation == "siblingOf":
                if gender == "Male":
                    relation_type = "Brother of:"
                if gender == "Female":
                    relation_type = "Sister of:"
            relation_statement = f"<li>{relation_type} <a href='/person-index/{relative_id}'>{relative_label}</a></li>"
            relations_section += relation_statement
        relations_section += "</ul>"
    else:
        relations_section = ""

    # English name
    name_en = person_data["name_en"]

    # Ancient Greek name
    name_grc = person_data["name_grc"]
    if name_grc == None:
        name_grc = ""
        
    person_page_html = person_page_template.format(
        name_en = name_en,
        name_grc = name_grc,
        external_id = external_id,
        person_link = "person-index/" + person_uri.split('/')[-1].split('#')[-1],
        gender = gender,
        roles_section = roles_section,
        speech_section = speech_section,
        author_section = author_section,
        real_or_fictional = real_or_fictional,
        relations_section = relations_section
        )
    path = "persons/"+ person_uri.split('/')[-1].split('#')[-1] +".html"
        
    with open(path, 'w', encoding='utf-8') as f:
        f.write(person_page_html)